![](https://github.com/datagong/data/blob/main/datagong.png?raw=true)

© DATAGONG - Tous droits réservés - 2026

📋 **Rappels Conditions Générales d'Utilisation**

⚠️ Les Notebooks sont privés

❌ partage des Notebooks, de leur contenu, des liens, des images, …

❌ publier les Notebooks sur GitHub (public) ou tout autre outil de versionning

✅ sauvegarder les Notebooks dans votre environnement personnel et privé

✅ réutiliser les codes dans le cadre de vos projets en entreprise / projets personnels / etc.

✅ annoter / modifier les Notebooks dans votre environnement personnel et privé

# <center><u><b>Data Visualization avec Streamlit et Plotly</b></u></center>

# <b>Streamlit + Plotly — 3. Plotly Express & Design System de graphes</b>

Dans le notebook précédent, vous avez connecté votre application à un vrai fichier CSV et mis en place le caching pour éviter de recharger les données à chaque interaction. Votre app sait désormais lire et afficher des données — il est temps de s'occuper de la façon dont elle les **montre**.

Quand vous construisez une application avec plusieurs graphiques, vous allez vite constater un problème : si chaque graphique est créé indépendamment, les styles (couleurs, marges, polices…) finissent par diverger. Le résultat fait "bricolé". Pour éviter cela, nous allons mettre en place ce qu'on appelle un **design system** : un ensemble de règles visuelles partagées par tous vos graphiques, encapsulées dans des fonctions réutilisables que nous rangerons dans `utils/charts.py`.

# 0. Définir un thème Plotly & fonctions utilitaires

L'idée est simple : plutôt que de configurer manuellement chaque graphique (marges, titre, template…), nous allons écrire deux fonctions — `make_line` et `make_bar` — qui produisent des figures Plotly déjà stylées. Tous les réglages communs sont centralisés dans une petite fonction interne `_apply_common`, ce qui vous garantit un rendu homogène sans effort.

Concrètement, voici les briques Plotly que nous utilisons dans le code ci-dessous :

### `px.line` et `px.bar`

Ce sont les deux fonctions principales de **Plotly Express** pour créer des graphiques. `px.line` trace des courbes, `px.bar` des diagrammes en barres. Leur signature est très proche : vous passez un DataFrame, le nom de la colonne en abscisse (`x`), celui en ordonnée (`y`), et éventuellement une colonne `color` pour différencier des catégories. Plotly Express se charge du reste — légende, axes, couleurs — et vous renvoie un objet `Figure` prêt à l'emploi.

### `fig.update_layout`

Une fois la figure créée, `update_layout` vous permet de modifier son apparence globale : template graphique, titre, marges, taille de police, etc. C'est cette méthode que nous appelons dans `_apply_common` pour appliquer le thème `plotly_white` et harmoniser les marges de toutes nos figures.

📖 Documentation : [`px.line`](https://plotly.com/python-api-reference/generated/plotly.express.line.html) · [`px.bar`](https://plotly.com/python-api-reference/generated/plotly.express.bar.html) · [`update_layout`](https://plotly.com/python-api-reference/generated/plotly.graph_objects.Figure.html#plotly.graph_objects.Figure.update_layout)

In [3]:
%%writefile ../streamlit_app/utils/charts.py
import plotly.express as px

# Définition du thème graphique commun
TEMPLATE = "plotly_white"

def _apply_common(fig, title: str):
    # Applique le template, le titre et les marges à la figure
    fig.update_layout(template=TEMPLATE, title=title, margin=dict(l=40, r=20, t=60, b=40))
    return fig

def make_line(df, x, y, color=None, title="Courbe"):
    # Crée un graphique en courbe avec Plotly Express
    fig = px.line(df, x=x, y=y, color=color)
    # Applique les paramètres communs
    return _apply_common(fig, title)

def make_bar(df, x, y, color=None, title="Barres"):
    # Crée un graphique en barres avec Plotly Express
    fig = px.bar(df, x=x, y=y, color=color, barmode="group")
    # Applique les paramètres communs
    return _apply_common(fig, title)


Overwriting ../streamlit_app/utils/charts.py


# 1. Utiliser ces fonctions dans `app.py`

Notre module `utils/charts.py` est prêt. Nous allons maintenant remplacer le graphique "en dur" du notebook précédent par un appel à nos nouvelles fonctions. Au passage, nous en profitons pour ajouter un premier **widget interactif** qui permettra à l'utilisateur de choisir le type de graphique affiché.

## 1.1 Les `selectbox` dans Streamlit

### `st.selectbox`

`st.selectbox` affiche une **liste déroulante** dans votre application. L'utilisateur clique, choisit une option, et la valeur sélectionnée est renvoyée dans une variable Python — c'est aussi simple que cela. Streamlit se charge de tout le rendu HTML côté navigateur.

```python
import streamlit as st

option = st.selectbox("Votre question ici", ["Option 1", "Option 2", "Option 3"])
st.write("Vous avez choisi :", option)
```

Le premier argument est le texte affiché au-dessus de la liste (le *label*), le second est la liste des choix. La variable `option` contient à tout moment la valeur sélectionnée par l'utilisateur.

Dans le code ci-dessous, nous utilisons une `selectbox` pour laisser l'utilisateur choisir entre un graphique en courbe et un graphique en barres. Selon sa sélection, nous appelons `make_line` ou `make_bar` — nos fonctions appliquent automatiquement le bon style, et le graphique se met à jour instantanément dans l'interface.

💡 Les widgets comme `st.selectbox` sont au cœur de Streamlit : chaque interaction de l'utilisateur relance le script Python de haut en bas, et Streamlit redessine uniquement ce qui a changé. Vous n'avez aucun **callback** à écrire — un callback, c'est une fonction que vous devez définir pour dire au framework "quand l'utilisateur clique ici, exécute ce code". Dans des frameworks comme **Dash**, par exemple, chaque interaction nécessite d'écrire explicitement ces callbacks. Streamlit vous épargne cette mécanique : le flux est entièrement déclaratif, vous écrivez votre logique de haut en bas et le framework s'occupe du reste.

📖 Documentation : [`st.selectbox`](https://docs.streamlit.io/develop/api-reference/widgets/st.selectbox)

In [2]:
%%writefile -a ../streamlit_app/app.py
# --- Section: Graphiques réutilisables ---
from utils.charts import make_line, make_bar

# Affiche une selectbox pour choisir le type de graphique
choix = st.selectbox("Type de graphique", ["Courbe", "Barres"])

# Selon le choix, génère la figure correspondante avec le style cohérent
if choix == "Courbe":
    fig = make_line(data, x="date", y="ventes", color="categorie",
                    title="Ventes — courbe")
else:
    fig = make_bar(data, x="date", y="ventes", color="categorie",
                   title="Ventes — barres")

# Affiche le graphique dans Streamlit
st.plotly_chart(fig, use_container_width=True)

Appending to ../streamlit_app/app.py


# <font color='#ff7373'><b>Félicitations !</b></font>

Vous avez posé les bases d'un vrai design system graphique : tous vos graphiques partagent désormais le même style, et ajouter une nouvelle visualisation se résume à un appel de fonction. C'est exactement cette approche qui fait la différence entre un prototype jetable et une application maintenable.

Dans le prochain notebook, nous irons plus loin dans l'**interactivité** en découvrant d'autres widgets Streamlit et la gestion de l'**état** (`st.session_state`) pour construire des interfaces plus riches.

# <center><font color='#3b4859'><u>![](https://github.com/datagong/data/blob/main/mini%20datagong%202.png?raw=true)</u></font></center>